In [1]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "10_dashboard_results"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


Mounted at /content/drive
PROC_PATH: /content/drive/MyDrive/TCC_USP/data_processed


In [ ]:
# 2. Carregar resultados de notebooks anteriores (JSON/CSV)
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json

# Arquivos de resultados gerados pelos notebooks anteriores
result_files = {
    "TF-IDF Models": os.path.join(PROC_PATH, "results_03_tfidf_models.json"),
    "Embeddings Models": os.path.join(PROC_PATH, "results_04_embeddings_models.json"),
    "LSTM Real": os.path.join(PROC_PATH, "results_09_lstm_real.json"),
    "TF-IDF Baselines (16)": os.path.join(PROC_PATH, "results_16_models_tfidf.json"),
}

results = {}

for source_name, filepath in result_files.items():
    if os.path.exists(filepath):
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Normalizar estrutura (diferentes notebooks podem ter formatos ligeiramente diferentes)
            if isinstance(data, dict):
                for model_name, metrics in data.items():
                    if isinstance(metrics, dict):
                        # Extrair AUC e MDA
                        if "auc" in metrics or "AUC" in metrics:
                            auc_key = "auc" if "auc" in metrics else "AUC"
                            mda_key = "mda" if "mda" in metrics else "MDA"
                            
                            # Lidar com estrutura aninhada (e.g., {"auc": {"value": 0.65}})
                            auc_val = metrics[auc_key]
                            mda_val = metrics[mda_key]
                            
                            if isinstance(auc_val, dict) and "value" in auc_val:
                                auc_val = auc_val["value"]
                            if isinstance(mda_val, dict) and "value" in mda_val:
                                mda_val = mda_val["value"]
                            
                            full_name = f"{model_name} ({source_name})"
                            results[full_name] = {
                                "AUC": float(auc_val) if auc_val else 0.0,
                                "MDA": float(mda_val) if mda_val else 0.0
                            }
        except Exception as e:
            print(f"⚠️ Erro ao carregar {filepath}: {e}")
    else:
        print(f"⚠️ Arquivo não encontrado: {filepath}")

if not results:
    print("\n❌ AVISO: Nenhum resultado carregado!")
    print("\nPara gerar resultados, execute primeiro:")
    print("  - Notebooks 03 (TF-IDF models)")
    print("  - Notebook 04 (Embeddings models)")
    print("  - Notebook 09 (LSTM real)")
    print("  - Notebook 16 (Models TF-IDF baselines)")
    print("\nUsando dados de exemplo para demonstração...")
    
    # Fallback com aviso explícito
    results = {
        "Baseline (exemplo)": {"AUC": 0.55, "MDA": 0.52},
        "TF-IDF (exemplo)": {"AUC": 0.66, "MDA": 0.58},
    }

df_res = pd.DataFrame(results).T.reset_index().rename(columns={"index": "Modelo"})
print(f"\n✅ {len(df_res)} modelos carregados")
display(df_res)


,Modelo,AUC,MDA
0,Baseline (Logit),0.55,0.52
1,TF-IDF (Logit),0.66,0.58
2,TF-IDF (XGBoost),0.50,0.58
3,Embeddings (Logit),0.56,0.58
4,Embeddings (RandomForest),0.44,0.66
5,LSTM,0.60,0.62


In [3]:
# 3. Tabela comparativa
fig_table = go.Figure(data=[go.Table(
    header=dict(values=list(df_res.columns), fill_color='lightblue', align='center'),
    cells=dict(values=[df_res[col] for col in df_res.columns], align='center')
)])
fig_table.update_layout(title="Comparação de Modelos - AUC e MDA")
fig_table.show()

In [4]:
# 4. Gráfico de barras comparando AUC e MDA
fig_bar = go.Figure()

fig_bar.add_trace(go.Bar(
    x=df_res["Modelo"], y=df_res["AUC"], name="AUC", marker_color="steelblue"
))
fig_bar.add_trace(go.Bar(
    x=df_res["Modelo"], y=df_res["MDA"], name="MDA", marker_color="darkorange"
))

fig_bar.update_layout(
    barmode="group",
    title="Desempenho dos Modelos (AUC vs MDA)",
    xaxis_title="Modelo",
    yaxis_title="Score"
)
fig_bar.show()

In [6]:
# 5. Visualização do Ibovespa com marcação de notícias
ibov_file = os.path.join(PROC_PATH, "ibovespa_clean.csv")
news_file = os.path.join(PROC_PATH, "noticias_real_clean.csv")

df_ibov = pd.read_csv(ibov_file)
df_news = pd.read_csv(news_file)

if "date" in df_ibov.columns: df_ibov.rename(columns={"date":"data"}, inplace=True)
if "date" in df_news.columns: df_news.rename(columns={"date":"data"}, inplace=True)

df_ibov["data"] = pd.to_datetime(df_ibov["data"])
df_news["data"] = pd.to_datetime(df_news["data"])

fig_ibov = go.Figure()
fig_ibov.add_trace(go.Scatter(
    x=df_ibov["data"], y=df_ibov["close"],
    mode="lines", name="IBOV", line=dict(color="blue")
))
fig_ibov.add_trace(go.Scatter(
    x=df_news["data"], y=[df_ibov["close"].iloc[-1]]*len(df_news),
    mode="markers", name="Notícias",
    text=df_news["title"] if "title" in df_news.columns else df_news["clean_text"],
    marker=dict(color="red", size=8, symbol="circle")
))

fig_ibov.update_layout(title="IBOV com Marcação de Notícias", xaxis_title="Data", yaxis_title="Fechamento")
fig_ibov.show()